# 04 · Beyond: evaluation, DBSCAN, explaining clusters

**Lecture 2 · Blocks B7-B9 · Lecture + demo with three short tasks · about 30 minutes (B7 evaluation 12, B8 DBSCAN 10, B9 explaining clusters 10)**

**Learning goal:** We evaluate clusterings internally (silhouette) and externally (ARI), choose k and ε, see where DBSCAN beats k-means, and explain clusters with profiles and a small decision tree.

**What we do:**
1. Task 1 - `silhouette(D, labels)` by its formula (about 5 min)
2. Task 2 - read ε for DBSCAN from the k-distance plot (about 3 min)
3. Task 3 - read the surrogate tree and describe each cluster in words (about 3 min)

Everything else is given: run it, look at the plots, discuss. The tricky part is not the code but the judgement: which number to trust, which ε to pick, which clustering to explain. The notebook runs from top to bottom.

In [ ]:
DATASET = "penguins"  # or "wholesale", "auto_mpg", "students" (see 01_dataset.ipynb)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.cluster.hierarchy import fcluster
from scipy.cluster.hierarchy import linkage as scipy_linkage
from scipy.spatial.distance import pdist, squareform
from sklearn.cluster import DBSCAN, KMeans
from sklearn.datasets import make_moons
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.tree import DecisionTreeClassifier, export_text

import ddm

## Setup: load X, D and the clusterings from 02/03 (given)

In [ ]:
df = ddm.load_dataset(DATASET)
info = ddm.dataset_info(DATASET)

X = ddm.load_array(DATASET, "X")  # feature matrix saved by 02_distance.ipynb
D = ddm.load_array(DATASET, "D")  # distance matrix saved by 02_distance.ipynb
meta = ddm.load_json(DATASET, "meta")  # which rows, which types, ...

if X is None or D is None or meta is None:
    missing = [name for name, obj in [("X", X), ("D", D), ("meta", meta)] if obj is None]
    print(f"NOTICE: results of lecture01/02_distance.ipynb not found in {ddm.processed_dir()} (missing: {', '.join(missing)}).")
    print("        Using a BASELINE instead: z-scored numeric columns + Euclidean distance (scipy pdist).")
    print("        Solve lecture01/02_distance.ipynb and run it to the end to work with your own X and D.")
    numeric_cols = [
        c for c in df.columns
        if pd.api.types.is_numeric_dtype(df[c]) and c not in info.label_candidates
    ]
    base = df[numeric_cols].dropna()
    X = ((base - base.mean()) / base.std()).to_numpy()
    D = squareform(pdist(X, metric="euclidean"))
    meta = {
        "rows": base.index.tolist(),
        "types": {c: "numeric" for c in numeric_cols},
        "log_columns": [],
        "feature_names": numeric_cols,
        "source": "baseline",
    }
else:
    print(f"Loaded X {X.shape} and D {D.shape} from {ddm.processed_dir()} (saved by 02_distance.ipynb).")

data = df.loc[meta["rows"]]  # the objects (rows) that X and D describe, in the same order
TYPES = meta["types"]
print(f"{len(data)} objects, {X.shape[1]} features in X.")

In [ ]:
K = 3  # number of clusters (penguins: 3 species)
LABEL = info.label_candidates[0]
truth = data[LABEL].fillna("missing").astype(str).to_numpy()

saved = ddm.load_json(DATASET, "labels")  # saved by 03_clustering.ipynb
if saved is not None and saved["rows"] == meta["rows"]:
    clusterings = {name: np.asarray(lab) for name, lab in saved["labelings"].items()}
    print(f"Loaded {len(clusterings)} clusterings from 03_clustering.ipynb: {list(clusterings)}")
else:
    print("NOTICE: no clusterings from 03_clustering.ipynb for these objects.")
    print("        Using BASELINE clusterings: SciPy average linkage on D and scikit-learn KMeans on X.")
    clusterings = {
        f"average linkage, SciPy (k={K})": fcluster(scipy_linkage(squareform(D, checks=False), "average"), K, "maxclust") - 1,
        f"k-means, scikit-learn (k={K})": KMeans(n_clusters=K, n_init=10, random_state=0).fit_predict(X),
    }

# Part 1 · B7 Evaluation

* **Internal** evaluation uses only the data: are clusters compact and well separated?
  Example: the silhouette coefficient.
* **External** evaluation compares with known labels (our label candidates): did the clustering
  find *that* structure? Example: the Adjusted Rand Index (ARI).

## Task 1 · The silhouette coefficient (about 5 minutes)

For every object $i$:

* $a(i)$ = mean distance of $i$ to the **other** members of its own cluster,
* $b(i)$ = the smallest mean distance of $i$ to the members of **another** cluster,
* $s(i) = \dfrac{b(i) - a(i)}{\max\big(a(i), b(i)\big)} \in [-1, 1]$, and $s(i) = 0$ if $i$ is alone in its cluster.

The silhouette coefficient is the mean of all $s(i)$. Close to 1: compact and well separated.
Around 0: clusters overlap. Negative: objects sit in the wrong cluster.

<details>
<summary><b>Hint</b> (click to open)</summary>

- `own = labels == labels[i]` is a boolean mask; `D[i, own]` are the distances from $i$ to its
  cluster *including itself* (distance 0), so divide the sum by `own.sum() - 1`.
- For $b(i)$: loop over `clusters`, skip `labels[i]`, take `D[i, labels == c].mean()`, keep the minimum.
</details>

In [ ]:
def silhouette(D: np.ndarray, labels: np.ndarray) -> float:
    """Mean silhouette coefficient of a clustering, from a distance matrix.

    Parameters
    ----------
    D : array of shape (n, n), distance matrix
    labels : array of shape (n,), cluster label of every object (at least 2 clusters)

    Returns
    -------
    float in [-1, 1]: the mean of s(i) over all objects
    """
    D = np.asarray(D, dtype=float)
    labels = np.asarray(labels)
    clusters = np.unique(labels)
    s = np.zeros(len(labels))
    for i in range(len(labels)):
        own = labels == labels[i]  # members of i's cluster (including i)
        if own.sum() == 1:
            continue  # alone in its cluster: s(i) = 0 by definition
        # TODO (a): a = mean distance from i to the OTHER members of its cluster
        # TODO (b): b = smallest mean distance from i to the members of another cluster
        # TODO (c): s[i] = (b - a) / max(a, b)
        raise NotImplementedError("silhouette")
    return float(s.mean())

In [ ]:
# Check Task 1 (compares with sklearn.metrics.silhouette_score)
def test_silhouette():
    pts = np.array([0.0, 1.0, 10.0, 12.0, 30.0])
    D_small = np.abs(pts[:, None] - pts[None, :])
    lab_small = np.array([0, 0, 1, 1, 2])  # cluster 2 has a single member
    got = silhouette(D_small, lab_small)
    expected = silhouette_score(D_small, lab_small, metric="precomputed")
    assert np.isclose(got, expected), f"small example: got {got:.4f}, scikit-learn says {expected:.4f}"
    for name, lab in clusterings.items():
        got = silhouette(D, lab)
        expected = silhouette_score(D, lab, metric="precomputed")
        assert np.isclose(got, expected), f"{name}: got {got:.4f}, scikit-learn says {expected:.4f}"


ddm.check("silhouette", test_silhouette)

### Evaluation table (given)

For the rest of the notebook we use scikit-learn's `silhouette_score` (same formula, faster).
DBSCAN marks noise with the label -1; noise objects are left out of the silhouette and
count as one extra group for the ARI.

In [ ]:
def evaluate(labels) -> dict:
    """Silhouette on D (without noise) and ARI against every label candidate."""
    labels = np.asarray(labels)
    clustered = labels != -1
    row = {"clusters": len(set(labels[clustered].tolist())), "noise": int((~clustered).sum())}
    if row["clusters"] >= 2:
        row["silhouette"] = silhouette_score(D[np.ix_(clustered, clustered)], labels[clustered], metric="precomputed")
    else:
        row["silhouette"] = np.nan
    for candidate in info.label_candidates:
        row[f"ARI vs {candidate}"] = adjusted_rand_score(data[candidate].fillna("missing").astype(str).to_numpy(), labels)
    return row


def evaluation_table(clusterings: dict) -> pd.DataFrame:
    table = pd.DataFrame({name: evaluate(lab) for name, lab in clusterings.items()}).T
    return table.astype({"clusters": int, "noise": int})


evaluation_table(clusterings).round(3)

### How many clusters? Elbow and silhouette (given)

We let k run from 2 to 8. The **elbow**: the SSE of k-means always drops with k; we look for
the k after which it drops much more slowly. The **silhouette** has a maximum instead.
Both are hints, not proofs.

In [ ]:
k_values = range(2, 9)
rows = []
Z_avg = scipy_linkage(squareform(D, checks=False), method="average")
for k in k_values:
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    avg_labels = fcluster(Z_avg, k, criterion="maxclust") - 1
    rows.append({
        "k": k,
        "k-means SSE (X)": km.inertia_,
        "silhouette k-means (X, Euclidean)": silhouette_score(X, km.labels_),
        "silhouette average linkage (D)": silhouette_score(D, avg_labels, metric="precomputed"),
    })
choose_k = pd.DataFrame(rows).set_index("k")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(choose_k.index, choose_k["k-means SSE (X)"], marker="o", color="#2a6fdb")
axes[0].set_xlabel("k")
axes[0].set_ylabel("SSE")
axes[0].set_title("Elbow: k-means SSE on X")
for col, color in [("silhouette k-means (X, Euclidean)", "#2a6fdb"), ("silhouette average linkage (D)", "#e8743b")]:
    axes[1].plot(choose_k.index, choose_k[col], marker="o", label=col, color=color)
axes[1].set_xlabel("k")
axes[1].set_ylabel("silhouette")
axes[1].set_title("Silhouette per k")
axes[1].legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()
choose_k.round(3)

# Part 2 · B8 DBSCAN

## DBSCAN: clusters = dense regions, the rest is noise

Two parameters: a radius $\varepsilon$ and `min_samples`. An object is a **core** object if at
least `min_samples` objects (itself included) lie within distance $\varepsilon$. Core objects
that reach each other form a cluster; objects in reach of a core object are **border**
objects; everything else is **noise** (label -1). No k needed, any cluster shape possible.

**Choosing ε with the k-distance plot:** for every object we take the distance to its
`min_samples - 1`-th nearest neighbour and sort these values. Inside clusters the values are
small; at the "knee", where the curve starts to rise steeply, we leave the dense regions.

In [ ]:
MIN_SAMPLES = 5  # rule of thumb: at least 4; larger values give fewer, denser clusters

# column 0 of every sorted row is the object itself (distance 0)
k_distance = np.sort(np.sort(D, axis=1)[:, MIN_SAMPLES - 1])

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(k_distance, color="#2a6fdb")
ax.set_xlabel("objects, sorted by their k-distance")
ax.set_ylabel(f"distance to neighbour no. {MIN_SAMPLES - 1}")
ax.set_title(f"k-distance plot ({DATASET}, min_samples = {MIN_SAMPLES})")
ax.grid(alpha=0.3)
plt.show()
print("Quantiles of the k-distance:", np.round(np.quantile(k_distance, [0.5, 0.8, 0.9, 0.95, 0.99]), 3))

## Task 2 · Choose ε (about 3 minutes)

Read the height of the knee from the plot above and set `EPS`. Then run the next cells and
look at the sensitivity table: how much does the result change if ε is a bit smaller or larger?

In [ ]:
# TODO: the height of the knee in the k-distance plot
EPS = None

if EPS is None:
    raise NotImplementedError("Task 2: set EPS to a value read from the k-distance plot")

In [ ]:
if EPS is None:
    print("NOTICE: EPS is not set yet (Task 2) - DBSCAN is skipped.")
else:
    labels_dbscan = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES, metric="precomputed").fit_predict(D)
    clusterings[f"DBSCAN (eps={EPS}, min_samples={MIN_SAMPLES})"] = labels_dbscan
    result = evaluate(labels_dbscan)
    print(f"DBSCAN with eps = {EPS}: {result['clusters']} clusters, {result['noise']} noise objects, "
          f"silhouette {result['silhouette']:.3f}, ARI vs {LABEL} {result[f'ARI vs {LABEL}']:.3f}")

    sensitivity = []
    for factor in (0.5, 0.75, 1.0, 1.5, 2.0):
        eps = EPS * factor
        lab = DBSCAN(eps=eps, min_samples=MIN_SAMPLES, metric="precomputed").fit_predict(D)
        sensitivity.append({"eps": round(eps, 4), **evaluate(lab)})
    display(pd.DataFrame(sensitivity).set_index("eps").round(3))

## All clusterings so far (given)

The evaluation table and one MDS picture with every clustering from 03 and DBSCAN.

In [ ]:
display(evaluation_table(clusterings).round(3))
ddm.plot_clusters_mds(D, {f"{LABEL} (known groups)": truth, **clusterings},
                      title=f"{DATASET}: all clusterings so far (MDS of D)")
plt.show()

## Where k-means fails: two half moons (given)

k-means assumes compact, roughly round clusters around a mean. Two interleaved half moons
break that assumption. Which methods find them?

In [ ]:
moons, moon_truth = make_moons(n_samples=300, noise=0.06, random_state=0)
moon_results = {
    "k-means (k=2)": KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(moons),
    "average linkage (k=2)": fcluster(scipy_linkage(moons, method="average"), 2, "maxclust") - 1,
    "single linkage (k=2)": fcluster(scipy_linkage(moons, method="single"), 2, "maxclust") - 1,
    "DBSCAN (eps=0.2, min_samples=5)": DBSCAN(eps=0.2, min_samples=5).fit_predict(moons),
}
titled = {}
for name, lab in moon_results.items():
    ari = adjusted_rand_score(moon_truth, lab)
    print(f"{name:34s} ARI vs true moons: {ari:.3f}")
    titled[f"{name}, ARI {ari:.2f}"] = lab
ddm.plot_clusters_2d(moons, titled, title="Two half moons")
plt.show()

## The village map from the warm-up, clustered by machines (given)

Remember the map from B2? Here are the same 24 buildings. First only by position (x, y in km).
Then we add our domain knowledge "the river separates": the bank (west/east) becomes an
extra attribute, and crossing the river costs as much as 3 km of walking.

In [ ]:
village = ddm.load_toy_village()
xy = village[["x", "y"]].to_numpy()
RIVER_COST_KM = 3.0
xy_river = np.column_stack([xy, RIVER_COST_KM * (village["bank"] == "east").to_numpy()])


def village_clusterings(F: np.ndarray) -> dict:
    """The same five algorithms on a village feature matrix F."""
    D_village = squareform(pdist(F))
    return {
        "k-means (k=4)": KMeans(n_clusters=4, n_init=10, random_state=0).fit_predict(F),
        "k-means (k=5)": KMeans(n_clusters=5, n_init=10, random_state=0).fit_predict(F),
        "single linkage (k=5)": fcluster(scipy_linkage(F, method="single"), 5, "maxclust") - 1,
        "complete linkage (k=5)": fcluster(scipy_linkage(F, method="complete"), 5, "maxclust") - 1,
        "DBSCAN (eps=1.5, min_samples=2)": DBSCAN(eps=1.5, min_samples=2, metric="precomputed").fit_predict(D_village),
    }


ddm.plot_toy_village(village_clusterings(xy), title="Village map: position only (x, y)")
plt.show()
ddm.plot_toy_village(village_clusterings(xy_river),
                     title=f"Village map: position + river bank (crossing costs {RIVER_COST_KM:.0f} km)")
plt.show()

Things to look for: the isolated farm X (own cluster? noise?), the row of farms along the
road (one long cluster or pieces?), and the four houses N-Q at the river (one group, or two
once the river counts?). Compare with the photos from the warm-up.

# Part 3 · Explaining clusters

A clustering is only useful if we can say *what* the clusters are. Two tools:

1. **Cluster profiles:** per cluster the mean of every numeric attribute and the most
   frequent value of every categorical attribute, next to the overall value.
2. **Surrogate decision tree:** a small tree (depth ≤ 3) that predicts the cluster label from
   the *original, interpretable* attributes. Its rules describe the clusters in words.
   **Fidelity** = share of objects where the tree agrees with the clustering.
   High fidelity at small depth = the clusters have a simple description.

In [ ]:
EXPLAIN = "k-medoids"  # (part of) the name of the clustering we want to explain

matches = [name for name in clusterings if EXPLAIN in name]
explain_name = matches[0] if matches else list(clusterings)[0]
explain_labels = np.asarray(clusterings[explain_name])
print(f"We explain: {explain_name}  (cluster sizes {np.bincount(explain_labels[explain_labels >= 0]).tolist()})")

In [ ]:
def cluster_profile(data: pd.DataFrame, labels, types: dict) -> pd.DataFrame:
    """Per cluster: mean (numeric/ordinal) or most frequent value with its share (nominal/binary)."""
    labels = pd.Series(np.asarray(labels), index=data.index)
    groups = sorted(labels.unique())
    table = {}
    for col, kind in types.items():
        if kind in ("numeric", "ordinal"):
            values = data[col].astype(float)
            row = {f"cluster {g}": values[labels == g].mean() for g in groups}
            row["all"] = values.mean()
            table[col] = {k: f"{v:.4g}" for k, v in row.items()}
        elif kind in ("nominal", "binary"):
            row = {}
            for g in groups + ["all"]:
                part = data[col] if g == "all" else data.loc[labels == g, col]
                counts = part.value_counts(normalize=True)
                row[g if g == "all" else f"cluster {g}"] = f"{counts.index[0]} ({counts.iloc[0]:.0%})" if len(counts) else "-"
            table[col] = row
    table["size"] = {**{f"cluster {g}": int((labels == g).sum()) for g in groups}, "all": len(labels)}
    return pd.DataFrame(table).T


cluster_profile(data, explain_labels, TYPES)

In [ ]:
def interpretable_features(data: pd.DataFrame, types: dict) -> pd.DataFrame:
    """Original units (no z-score, no log) so that the tree rules are readable."""
    columns = {}
    for col, kind in types.items():
        if kind in ("numeric", "ordinal"):
            values = data[col].astype(float)
            columns[col] = values.fillna(values.median())
        elif kind in ("nominal", "binary"):
            for value in sorted(data[col].dropna().unique()):
                columns[f"{col}={value}"] = (data[col] == value).astype(int)
    return pd.DataFrame(columns, index=data.index)


F = interpretable_features(data, TYPES)
for depth in (1, 2, 3):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(F, explain_labels)
    fidelity = (tree.predict(F) == explain_labels).mean()
    print(f"depth {depth}: fidelity {fidelity:.1%}")

tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(F, explain_labels)
print()
print(export_text(tree, feature_names=list(F.columns), class_names=[f"cluster {c}" for c in tree.classes_]))

## Task 3 · Read the tree (about 3 minutes)

Describe every cluster in one sentence, using only the tree rules and the profile table.
Which attributes does the tree use, and does that fit the question of our dataset?

*Our answer:*

- cluster 0: ...
- cluster 1: ...
- cluster 2: ...

## Debrief questions

1. Which clustering has the best silhouette, which the best ARI against the known groups?
   Are they the same? Why not?
2. DBSCAN: how sensitive is the result to ε? What happens to the noise?
3. The village: which algorithm reproduces which group photo from the warm-up?
4. Which attributes does the surrogate tree use?